<a href="https://colab.research.google.com/github/vpp1004/GESPRO_GESTIONTAREAS/blob/master/Pr%C3%A1ctica_gesti%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 1 · Primer contacto con LangChain

**Sistemas de Gestión de la Información Clínica** · Grado en Ingeniería de la Salud

En la presentación hemos visto qué hace un modelo de lenguaje: recibe texto y predice, token a token, el texto que viene después. En este notebook vamos a usar un LLM a través de **LangChain** para resolver un problema muy habitual en información clínica: **convertir texto clínico libre en información estructurada y útil**.

El código ya está escrito. Tu trabajo consiste en **ejecutarlo, modificar los prompts y observar qué cambia**. Las celdas marcadas con ✏️ son las que debes tocar.

| Bloque | Qué aprenderás |
|---|---|
| 1. Llamada básica | Mensajes de sistema y de usuario; qué devuelve el modelo |
| 2. Plantillas de prompts | Reutilizar un mismo prompt con variables |
| 3. Temperatura | Por qué el mismo prompt no siempre da la misma respuesta |
| 4. Salida estructurada | Extraer diagnósticos, negaciones y antecedentes familiares con un esquema |
| 5. Cadenas y lotes | Procesar 10 informes y **medir** cuántos diagnósticos acierta el modelo |

Lo que programes aquí reaparecerá en las sesiones 2 y 3 como componentes visuales de Langflow.

> ⚠️ **Datos.** Todos los informes de este notebook son **sintéticos**. Nunca envíes datos reales de pacientes a un servicio externo: los datos de salud son una categoría especial en el RGPD.

## 0. Preparación del entorno

Instalamos las librerías. Si trabajas en Google Colab o en un entorno nuevo, ejecuta la celda siguiente (solo hace falta una vez).

### Entorno local (pc)

In [ ]:
#%pip install -q -U langchain langchain-openai langchain-google-genai pydantic pandas

### Gggole Colab

In [ ]:
%pip install -q -U langchain langchain-openai langchain-google-genai

### Elegir proveedor y clave de API

Puedes trabajar con dos proveedores:

- **OpenAI**, con la clave proporcionada en clase (tiene un límite de gasto). Modelo recomendado: `gpt-4.1-nano`, un modelo pequeño y barato **sin razonamiento**.
- **Google**, con tu propia cuenta en el plan gratuito de Google AI Studio (<https://aistudio.google.com/apikey>). El plan gratuito limita el número de peticiones por minuto, así que algunas celdas pueden ir más lentas.

✏️ Cambia `PROVEEDOR` si vas a usar Google. La clave se pide de forma oculta y **no se guarda en el notebook**: no la escribas nunca directamente en una celda. En Colab las claves se gurdan en secretos 🔑 (barra lateral)

In [ ]:
PROVEEDOR = "google"        # "openai" o "google"
MODELOS = {
    "openai": "gpt-4o-mini",
    "google": "gemini-3.5-flash-lite",   # si no está disponible, consulta los modelos del plan gratuito en AI Studio
}
VARIABLE_CLAVE = {"openai": "OPENAI_API_KEY", "google": "GOOGLE_API_KEY"}

### Colab
Debes tener guardada la clave en secretos 🔑 (barra lateral)

In [ ]:
import os
from google.colab import userdata
os.environ[VARIABLE_CLAVE[PROVEEDOR]] = userdata.get(VARIABLE_CLAVE[PROVEEDOR])
print("Proveedor:", PROVEEDOR, "· Modelo:", MODELOS[PROVEEDOR])

Proveedor: google · Modelo: gemini-3.5-flash-lite


### En local
Debes tener guardada la clave localmente

In [ ]:
"""
import os
from getpass import getpass
if not os.environ.get(VARIABLE_CLAVE[PROVEEDOR]):
    os.environ[VARIABLE_CLAVE[PROVEEDOR]] = getpass(f"Introduce tu {VARIABLE_CLAVE[PROVEEDOR]}: ")

print("Proveedor:", PROVEEDOR, "· Modelo:", MODELOS[PROVEEDOR])
"""

'\nimport os\nfrom getpass import getpass\nif not os.environ.get(VARIABLE_CLAVE[PROVEEDOR]):\n    os.environ[VARIABLE_CLAVE[PROVEEDOR]] = getpass(f"Introduce tu {VARIABLE_CLAVE[PROVEEDOR]}: ")\n\nprint("Proveedor:", PROVEEDOR, "· Modelo:", MODELOS[PROVEEDOR])\n'

### Crear una instancia de un modelo

In [ ]:
from langchain.chat_models import init_chat_model

def crear_modelo(temperatura=0.0):
    # Devuelve un modelo de chat del proveedor elegido con la temperatura indicada.
    return init_chat_model(
        MODELOS[PROVEEDOR],
        model_provider="openai" if PROVEEDOR == "openai" else "google_genai",
        temperature=temperatura,
    )

llm = crear_modelo(temperatura=0)

### Los datos: 10 informes clínicos breves

Trabajaremos con diez fragmentos de informes clínicos sintéticos. Cada uno lleva una **gold standard** (etiqueta de referencia validada por expertos) con los diagnósticos que menciona y dos atributos:

- `negado`: el texto dice que el paciente **no** tiene ese diagnóstico ("se descarta neumonía").
- `antecedente_familiar`: el diagnóstico lo tiene **un familiar**, no el paciente ("madre con cáncer de mama").

Estos dos atributos son la causa más frecuente de errores al extraer información de texto clínico: un sistema que solo busca la palabra "neumonía" diría que el paciente tiene neumonía. La anotación la usaremos en el bloque 5 para evaluar el modelo.

In [ ]:
# Datos sintéticos
INFORMES = [
    {"id": 1,
     "texto": "Varón de 67 años con hipertensión arterial y diabetes mellitus tipo 2 que acude por dolor torácico. "
              "ECG sin alteraciones y troponinas normales; se descarta infarto agudo de miocardio. "
              "Padre fallecido por cáncer de colon.",
     "gold": [
         {"diagnostico": "hipertensión arterial", "negado": False, "antecedente_familiar": False},
         {"diagnostico": "diabetes mellitus tipo 2", "negado": False, "antecedente_familiar": False},
         {"diagnostico": "infarto agudo de miocardio", "negado": True, "antecedente_familiar": False},
         {"diagnostico": "cáncer de colon", "negado": False, "antecedente_familiar": True},
     ]},
    {"id": 2,
     "texto": "Mujer de 45 años con antecedentes personales de asma bronquial. Consulta por tos y fiebre de tres días. "
              "La radiografía de tórax no muestra condensaciones, por lo que se descarta neumonía. "
              "Juicio clínico: bronquitis aguda.",
     "gold": [
         {"diagnostico": "asma bronquial", "negado": False, "antecedente_familiar": False},
         {"diagnostico": "neumonía", "negado": True, "antecedente_familiar": False},
         {"diagnostico": "bronquitis aguda", "negado": False, "antecedente_familiar": False},
     ]},
    {"id": 3,
     "texto": "Mujer de 52 años con fibroadenoma de mama izquierda diagnosticado por ecografía hace dos años, "
              "estable en el control actual. Madre y tía materna con cáncer de mama.",
     "gold": [
         {"diagnostico": "fibroadenoma de mama", "negado": False, "antecedente_familiar": False},
         {"diagnostico": "cáncer de mama", "negado": False, "antecedente_familiar": True},
     ]},
    {"id": 4,
     "texto": "Varón de 30 años sin antecedentes de interés. Acude a urgencias por dolor en fosa ilíaca derecha y diarrea. "
              "Ecografía abdominal normal; se descarta apendicitis aguda. Diagnóstico al alta: gastroenteritis aguda.",
     "gold": [
         {"diagnostico": "apendicitis aguda", "negado": True, "antecedente_familiar": False},
         {"diagnostico": "gastroenteritis aguda", "negado": False, "antecedente_familiar": False},
     ]},
    {"id": 5,
     "texto": "Mujer de 78 años con fibrilación auricular en tratamiento anticoagulante e insuficiencia cardiaca crónica. "
              "Ingresa por descompensación de su insuficiencia cardiaca. Hermano con enfermedad de Alzheimer.",
     "gold": [
         {"diagnostico": "fibrilación auricular", "negado": False, "antecedente_familiar": False},
         {"diagnostico": "insuficiencia cardiaca crónica", "negado": False, "antecedente_familiar": False},
         {"diagnostico": "enfermedad de Alzheimer", "negado": False, "antecedente_familiar": True},
     ]},
    {"id": 6,
     "texto": "Niño de 8 años con dermatitis atópica desde la lactancia. Su padre tiene psoriasis. "
              "En la exploración no presenta signos de infección cutánea.",
     "gold": [
         {"diagnostico": "dermatitis atópica", "negado": False, "antecedente_familiar": False},
         {"diagnostico": "psoriasis", "negado": False, "antecedente_familiar": True},
         {"diagnostico": "infección cutánea", "negado": True, "antecedente_familiar": False},
     ]},
    {"id": 7,
     "texto": "Varón de 59 años con EPOC y obesidad. Se solicita polisomnografía por sospecha de síndrome de apnea "
              "obstructiva del sueño. Sin antecedentes familiares de enfermedad cardiovascular.",
     "gold": [
         {"diagnostico": "EPOC", "negado": False, "antecedente_familiar": False},
         {"diagnostico": "obesidad", "negado": False, "antecedente_familiar": False},
         {"diagnostico": "síndrome de apnea obstructiva del sueño", "negado": False, "antecedente_familiar": False},
         {"diagnostico": "enfermedad cardiovascular", "negado": True, "antecedente_familiar": True},
     ]},
    {"id": 8,
     "texto": "Mujer de 34 años, gestante de 20 semanas. La sobrecarga oral de glucosa es normal, por lo que se descarta "
              "diabetes gestacional. Presenta anemia ferropénica. Madre con hipotiroidismo.",
     "gold": [
         {"diagnostico": "diabetes gestacional", "negado": True, "antecedente_familiar": False},
         {"diagnostico": "anemia ferropénica", "negado": False, "antecedente_familiar": False},
         {"diagnostico": "hipotiroidismo", "negado": False, "antecedente_familiar": True},
     ]},
    {"id": 9,
     "texto": "Varón de 71 años con cardiopatía isquémica e insuficiencia renal crónica estadio 3. "
              "No refiere antecedentes de ictus. Su hermana padece diabetes mellitus tipo 1.",
     "gold": [
         {"diagnostico": "cardiopatía isquémica", "negado": False, "antecedente_familiar": False},
         {"diagnostico": "insuficiencia renal crónica", "negado": False, "antecedente_familiar": False},
         {"diagnostico": "ictus", "negado": True, "antecedente_familiar": False},
         {"diagnostico": "diabetes mellitus tipo 1", "negado": False, "antecedente_familiar": True},
     ]},
    {"id": 10,
     "texto": "Mujer de 25 años que consulta por cefalea intensa. Exploración neurológica normal. Se descarta hemorragia "
              "subaracnoidea mediante TC craneal. Juicio clínico: migraña sin aura. Antecedentes familiares de migraña "
              "por línea materna.",
     "gold": [
         {"diagnostico": "hemorragia subaracnoidea", "negado": True, "antecedente_familiar": False},
         {"diagnostico": "migraña sin aura", "negado": False, "antecedente_familiar": False},
         {"diagnostico": "migraña", "negado": False, "antecedente_familiar": True},
     ]},
]

# Sinónimos aceptados al comparar con la referencia (el modelo puede desarrollar siglas, etc.)
SINONIMOS = {
    "EPOC": ["enfermedad pulmonar obstructiva crónica"],
    "síndrome de apnea obstructiva del sueño": ["apnea obstructiva del sueño", "SAOS", "SAHS", "apnea del sueño"],
    "ictus": ["accidente cerebrovascular", "enfermedad cerebrovascular", "ACV"],
    "enfermedad de Alzheimer": ["Alzheimer"],
    "insuficiencia cardiaca crónica": ["insuficiencia cardíaca crónica"],
}

print(f"{len(INFORMES)} informes, {sum(len(i['gold']) for i in INFORMES)} diagnósticos anotados")

10 informes, 31 diagnósticos anotados


In [ ]:
import pandas as pd

pd.DataFrame(INFORMES)[['id','texto']].to_csv("informes.csv", index=False)

In [ ]:
import pandas as pd

# Desglosamos la lista de diagnósticos 'gold' manteniendo los datos del informe correspondiente
df_desglosado = pd.json_normalize(
    INFORMES,
    record_path='gold',
    meta=['id', 'texto'],
    record_prefix='gold_'
)

# Reordenamos las columnas para mayor claridad
df_desglosado = df_desglosado[['id', 'texto', 'gold_diagnostico', 'gold_negado', 'gold_antecedente_familiar']]
df_desglosado.to_csv("informes.csv", index=False)

In [ ]:
pd.DataFrame(INFORMES)

,id,texto,gold
0,1,Varón de 67 años con hipertensión arterial y d...,"[{'diagnostico': 'hipertensión arterial', 'neg..."
1,2,Mujer de 45 años con antecedentes personales d...,"[{'diagnostico': 'asma bronquial', 'negado': F..."
2,3,Mujer de 52 años con fibroadenoma de mama izqu...,"[{'diagnostico': 'fibroadenoma de mama', 'nega..."
3,4,Varón de 30 años sin antecedentes de interés. ...,"[{'diagnostico': 'apendicitis aguda', 'negado'..."
4,5,Mujer de 78 años con fibrilación auricular en ...,"[{'diagnostico': 'fibrilación auricular', 'neg..."
5,6,Niño de 8 años con dermatitis atópica desde la...,"[{'diagnostico': 'dermatitis atópica', 'negado..."
6,7,Varón de 59 años con EPOC y obesidad. Se solic...,"[{'diagnostico': 'EPOC', 'negado': False, 'ant..."
7,8,"Mujer de 34 años, gestante de 20 semanas. La s...","[{'diagnostico': 'diabetes gestacional', 'nega..."
8,9,Varón de 71 años con cardiopatía isquémica e i...,"[{'diagnostico': 'cardiopatía isquémica', 'neg..."
9,10,Mujer de 25 años que consulta por cefalea inte...,"[{'diagnostico': 'hemorragia subaracnoidea', '..."


---
## Bloque 1 · Llamada básica al modelo

Un modelo de chat recibe una lista de **mensajes**. Los dos tipos más importantes son:

- **Mensaje de sistema** (`SystemMessage`): instrucciones generales sobre el papel del modelo y cómo debe responder. El usuario final normalmente no lo ve.
- **Mensaje de usuario** (`HumanMessage`): la petición concreta.

El modelo devuelve un `AIMessage`. Entra texto y sale texto.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
from IPython.display import display, Markdown

mensajes = [
    SystemMessage(content="Eres un asistente que ayuda a profesionales sanitarios. Responde en español, "
                          "de forma breve y sin añadir información que no aparezca en el texto."),
    HumanMessage(content="Resume en una frase el siguiente informe:\n\n" + INFORMES[0]["texto"]),
]

respuesta = llm.invoke(mensajes)

# Extraemos el texto de forma segura si viene formateado en bloque de contenido de Google
contenido_texto = respuesta.content
if isinstance(contenido_texto, list) and len(contenido_texto) > 0:
    if isinstance(contenido_texto[0], dict) and 'text' in contenido_texto[0]:
        contenido_texto = contenido_texto[0]['text']

display(Markdown(str(contenido_texto)))
print("Uso de tokens:   ", respuesta.usage_metadata)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Varón de 67 años con hipertensión arterial y diabetes mellitus tipo 2 que consulta por dolor torácico, en el que se descarta infarto agudo de miocardio.

Uso de tokens:    {'input_tokens': 95, 'output_tokens': 40, 'total_tokens': 135, 'input_token_details': {'cache_read': 0}}


### Qué hay dentro de la respuesta

Además del texto, la respuesta trae metadatos: qué modelo respondió, por qué terminó de generar y **cuántos tokens** se consumieron en la entrada y en la salida. Los tokens son lo que se factura y lo que ocupa la ventana de contexto.

In [ ]:
print("Tipo de objeto:  ", type(respuesta).__name__)
print("Contenido:       ", respuesta.content[:80], "...")
print("Uso de tokens:   ", respuesta.usage_metadata)
print("Metadatos:       ", {k: v for k, v in respuesta.response_metadata.items() if k in ("model_name", "finish_reason")})

Tipo de objeto:   AIMessage
Contenido:        [{'type': 'text', 'text': 'Varón de 67 años con hipertensión arterial y diabetes mellitus tipo 2 que consulta por dolor torácico, en el que se descarta infarto agudo de miocardio.', 'extras': {'signature': 'EmAKXgFpFH0TFK5SJUMZgUsnvhuzqqhO0jsOdO+YLCbXVVHnOKOdZ/LpUXWuacPtmw8I14KG6QPn6u59JFiO03jYM1Jd1O0U1c+PHZAsBOC/vtF+zsc6AgTwRdheJltf9aQ='}}] ...
Uso de tokens:    {'input_tokens': 95, 'output_tokens': 40, 'total_tokens': 135, 'input_token_details': {'cache_read': 0}}
Metadatos:        {'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite'}


✏️ **Prueba tú.** Modifica el mensaje de sistema de la celda anterior y vuelve a ejecutar:

1. Pide que responda "como si hablara con el paciente, sin tecnicismos".
2. Pide que responda **solo** con una lista de diagnósticos separados por comas.
3. Elimina el mensaje de sistema. ¿Cambia la longitud de la respuesta? ¿Y el número de tokens de salida?

In [ ]:
mensajes = [
    SystemMessage(content="Eres un asistente que ayuda a profesionales sanitarios. Responde en español, "
                          "de forma breve y sin añadir información que no aparezca en el texto."),
    HumanMessage(content="Resume en una frase el siguiente informe,pero como si hablara con el paciente, sin tecnicismos:\n\n" + INFORMES[0]["texto"]),
]

respuesta = llm.invoke(mensajes)

# Extraemos el texto de forma segura si viene formateado en bloque de contenido de Google
contenido_texto = respuesta.content
if isinstance(contenido_texto, list) and len(contenido_texto) > 0:
    if isinstance(contenido_texto[0], dict) and 'text' in contenido_texto[0]:
        contenido_texto = contenido_texto[0]['text']

display(Markdown(str(contenido_texto)))

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Usted no ha sufrido un infarto, aunque seguiremos vigilando su dolor en el pecho teniendo en cuenta su presión alta, su diabetes y los antecedentes familiares.

In [ ]:
mensajes = [
    SystemMessage(content="Eres un asistente que ayuda a profesionales sanitarios. Responde en español, "
                          "de forma breve y sin añadir información que no aparezca en el texto."),
    HumanMessage(content="Responde solo con una lista de diagnósticos separados por comas:\n\n" + INFORMES[0]["texto"]),
]

respuesta = llm.invoke(mensajes)

# Extraemos el texto de forma segura si viene formateado en bloque de contenido de Google
contenido_texto = respuesta.content
if isinstance(contenido_texto, list) and len(contenido_texto) > 0:
    if isinstance(contenido_texto[0], dict) and 'text' in contenido_texto[0]:
        contenido_texto = contenido_texto[0]['text']

display(Markdown(str(contenido_texto)))

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Hipertensión arterial, diabetes mellitus tipo 2, dolor torácico

In [ ]:
mensajes = [
    HumanMessage(content="Resume en una frase el siguiente informe,pero como si hablara con el paciente, sin tecnicismos:\n\n" + INFORMES[0]["texto"]),
]

respuesta = llm.invoke(mensajes)

# Extraemos el texto de forma segura si viene formateado en bloque de contenido de Google
contenido_texto = respuesta.content
if isinstance(contenido_texto, list) and len(contenido_texto) > 0:
    if isinstance(contenido_texto[0], dict) and 'text' in contenido_texto[0]:
        contenido_texto = contenido_texto[0]['text']

display(Markdown(str(contenido_texto)))
print("Uso de tokens:   ", respuesta.usage_metadata)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


"Aunque viniste con dolor en el pecho, las pruebas descartaron un infarto, pero como tienes antecedentes familiares de cáncer de colon, debemos seguir vigilándote."

Uso de tokens:    {'input_tokens': 79, 'output_tokens': 34, 'total_tokens': 113, 'input_token_details': {'cache_read': 0}}


Sí cambia la longitud de la respuesta, ya que cada vez que lo ejecutas la IA te da una respuesta distinta.
El número de tokens disminuye de los 142 que teníamos originalmente a los 128 que tenemos ahora. Suponemos que al eliminar un prompt el número de tokens cambie.

---
## Bloque 2 · Plantillas de prompts

En una aplicación real no escribimos el prompt a mano para cada informe. Definimos una **plantilla** con huecos (`{informe}`, `{destinatario}`) y la rellenamos con datos distintos cada vez. En Langflow esto será el componente *Prompt Template*.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

plantilla_resumen = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente clínico. Redactas resúmenes fieles al informe original, en español, "
               "adaptados al destinatario. No inventes datos que no aparezcan en el informe."),
    ("human", "Destinatario: {destinatario}\n\nInforme:\n{informe}\n\nRedacta un resumen de un máximo de tres frases."),
])

# La plantilla rellenada es una lista de mensajes, igual que en el bloque 1
prompt_rellenado = plantilla_resumen.invoke({"informe": INFORMES[1]["texto"], "destinatario": "el paciente"})
for m in prompt_rellenado.messages:
    print(f"[{m.type}] {m.content}\n")

[system] Eres un asistente clínico. Redactas resúmenes fieles al informe original, en español, adaptados al destinatario. No inventes datos que no aparezcan en el informe.

[human] Destinatario: el paciente

Informe:
Mujer de 45 años con antecedentes personales de asma bronquial. Consulta por tos y fiebre de tres días. La radiografía de tórax no muestra condensaciones, por lo que se descarta neumonía. Juicio clínico: bronquitis aguda.

Redacta un resumen de un máximo de tres frases.



Ahora reutilizamos la misma plantilla con varios informes y dos destinatarios distintos.

In [ ]:
destinatarios = ["el paciente, sin tecnicismos", "su médico de atención primaria"]

for informe in INFORMES[1:3]:
    print(f"===== Informe {informe['id']} =====")
    for dest in destinatarios:
        prompt = plantilla_resumen.invoke({"informe": informe["texto"], "destinatario": dest})
        print(f"--> Para {dest}:\n{llm.invoke(prompt).content}\n")

===== Informe 2 =====


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


--> Para el paciente, sin tecnicismos:
[{'type': 'text', 'text': 'Consultó por tos y fiebre de tres días de evolución, contando con antecedentes de asma. Las pruebas de imagen descartaron una neumonía en sus pulmones. El diagnóstico final es una bronquitis aguda.', 'extras': {'signature': 'EmAKXgFpFH0TIX2E2V64fC7VvaGX0YMYL/md1rgLOojowdjapq/IixOTY/mRGwQUIAvZLkfVEz+CCIj94CsJKWcaxX9Os1wEOxZF37TO4tplTul2Na57Alg11Myc2hFnuR4='}}]



/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


--> Para su médico de atención primaria:
[{'type': 'text', 'text': 'Mujer de 45 años con antecedentes de asma bronquial que consulta por un cuadro de tres días de evolución de tos y fiebre. La exploración radiológica de tórax no muestra signos de condensación, descartándose neumonía. El juicio clínico establecido es de bronquitis aguda.', 'extras': {'signature': 'EmAKXgFpFH0TNHJJSBm+JF8NfAVE0Jm4wYHuqxXYowj7bB/kKOFnhaPB0yy/Yd91VMrC2fKrDL74n8GcXpL0PM7jyHPzNcY8QLzI/qIQraNXfsA2Z46DfRScjQiUTk6IbOw='}}]

===== Informe 3 =====


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


--> Para el paciente, sin tecnicismos:
[{'type': 'text', 'text': 'En tu control actual, se ha comprobado que el bulto benigno (fibroadenoma) de tu mama izquierda, que te diagnosticaron hace dos años, sigue exactamente igual y sin cambios. Dado que tienes antecedentes familiares de cáncer de mama en tu madre y tu tía, es muy importante que continúes acudiendo a tus revisiones médicas periódicas.', 'extras': {'signature': 'EmAKXgFpFH0Tl9UokDtwDk0MvPQJHIC0FUOQjP1sZeBdaEsM87i1UHL9+0DnT0Ndfp1M2swdD2NA3di3/3fvureAelEFu1b/oI7ivlsqSbf1mrrK66ICxjGeKnxnaEv9HPg='}}]



/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


--> Para su médico de atención primaria:
[{'type': 'text', 'text': 'Paciente mujer de 52 años con antecedentes familiares de cáncer de mama (madre y tía materna). Presenta un fibroadenoma en la mama izquierda diagnosticado por ecografía hace dos años, el cual se mantiene estable en el control actual. Se remite para su conocimiento y seguimiento desde atención primaria.', 'extras': {'signature': 'EmAKXgFpFH0TC2Sl5cGNSjaNKRtQ2obIY5g5JeDqpj3IKi2Wd0b7HnKag2g15efjMIGZR5cHBvhhme/OajhYLHpU6pArDginiHVzHQaJ4ApzGZajm4KWN2B281e1ISgvuH4='}}]



✏️ **Prueba tú.**

1. Añade un tercer destinatario (por ejemplo, "un servicio de codificación clínica" o "un familiar del paciente").
2. Añade una nueva variable a la plantilla, por ejemplo `{idioma}`, y úsala para pedir el resumen en inglés o en gallego.
3. Revisa los resúmenes para el paciente del informe 3: ¿queda claro que el cáncer de mama lo tuvieron la madre y la tía, y no la paciente?

In [ ]:
destinatarios = ["el paciente, sin tecnicismos", "su médico de atención primaria","un familiar del paciente,cercano"]

for informe in INFORMES[1:3]:
    print(f"===== Informe {informe['id']} =====")
    for dest in destinatarios:
        prompt = plantilla_resumen.invoke({"informe": informe["texto"], "destinatario": dest})
        print(f"--> Para {dest}:\n{llm.invoke(prompt).content}\n")

===== Informe 2 =====


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


--> Para el paciente, sin tecnicismos:
[{'type': 'text', 'text': 'Acudiste a la consulta por tos y fiebre de tres días de evolución, teniendo como antecedente asma. Tras realizarte una radiografía de tórax, se descartó la presencia de neumonía. El diagnóstico final es una bronquitis aguda.', 'extras': {'signature': 'EmAKXgFpFH0THN77Hg1m3nThHsQqIequAASdLnpVhXsdh4pv0lmdpI4Nqt0xSZdb3Caa1QPXIqdjMD6HbajtVln34NOyT+kLsOJXJkvqdrC05z5UhW05nTb0ZCy/ErjjAaU='}}]



/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


--> Para su médico de atención primaria:
[{'type': 'text', 'text': 'Paciente mujer de 45 años con antecedentes de asma bronquial que consulta por un cuadro de tres días de evolución de tos y fiebre. La exploración radiológica de tórax descarta la presencia de neumonía al no mostrar condensaciones. Se establece el juicio clínico de bronquitis aguda.', 'extras': {'signature': 'EmAKXgFpFH0TRykpsohm4VKjIUv8NekyyJeOR5CyRx9Q0kutbAfvQaV+hNA6NJ31uRjpXnwhro8n86eqx7/HDtpv2eBvmSyVmE+Ay7Kz8DAmtvHu8n1EzviRzlZwV1GQilY='}}]



/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


... (3 outputs left out)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


--> Para su médico de atención primaria:
[{'type': 'text', 'text': 'Paciente mujer de 52 años con antecedentes familiares de cáncer de mama (madre y tía materna). Presenta un fibroadenoma en la mama izquierda diagnosticado por ecografía hace dos años, el cual se mantiene estable en el control actual. Se remite esta información para su conocimiento y seguimiento en atención primaria.', 'extras': {'signature': 'EmAKXgFpFH0TO5qWGheP3fg9uCAtR5/a3PrIoG0naBibKDyD/sorRssRLsrDFgTNsr6mM/BhqvPTlm1AY8utNoZmdHC4RgEdMTGclrHx66OsjpEDxguU2b9gjRwK0akACMo='}}]



/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


--> Para un familiar del paciente,cercano:
[{'type': 'text', 'text': 'En el control actual, el bulto benigno (fibroadenoma) de la mama izquierda que te diagnosticaron hace dos años se mantiene estable y sin cambios. Teniendo en cuenta tus antecedentes familiares de cáncer de mama en tu madre y tu tía, es muy importante que continúes acudiendo a tus revisiones médicas periódicas.', 'extras': {'signature': 'EmAKXgFpFH0T40aAoXDDQT4wp9CQMhpJTAEi1CZsmRrey3rgZEhM+ZZtQ7ZCEOmLom3BcRsYhQCwmhylta1rr8+OCs2CFJm0LcCcRY8VECCSmpJdFdapROMc0W+1EWStuJU='}}]



In [ ]:
plantilla_resumen_idioma = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente clínico. Redactas resúmenes fieles al informe original, en el idioma que te indique, "
               "adaptados al destinatario. No inventes datos que no aparezcan en el informe."),
    ("human", "Destinatario: {destinatario}\n\nInforme:\n{informe}\n\nIdioma: {idioma}\n\nRedacta un resumen de un máximo de tres frases."),
])

# La plantilla rellenada es una lista de mensajes, igual que en el bloque 1
for idioma in ["inglés", "gallego","japonés","ruso","árabe"]:
    print(f"===== Resumen en {idioma} =====")
    prompt_rellenado_idioma = plantilla_resumen_idioma.invoke({
        "informe": INFORMES[1]["texto"],
        "destinatario": "el paciente",
        "idioma": idioma
    })
    respuesta_idioma = llm.invoke(prompt_rellenado_idioma)

    # Extraemos el texto de forma segura si viene formateado en bloque de contenido de Google
    contenido_texto = respuesta_idioma.content
    if isinstance(contenido_texto, list) and len(contenido_texto) > 0:
        if isinstance(contenido_texto[0], dict) and 'text' in contenido_texto[0]:
            contenido_texto = contenido_texto[0]['text']

    print(str(contenido_texto) + "\n")

===== Resumen en inglés =====


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


You consulted us due to a three-day history of cough and fever, given your personal history of bronchial asthma. A chest X-ray was performed and showed no signs of consolidation, ruling out pneumonia. Based on this, your diagnosis is acute bronchitis.

===== Resumen en gallego =====


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Acudiu a consulta por to e febre de tres días de evolución, tendo vostede antecedentes de asma bronquial. A radiografía de tórax realizada descartou a presenza de pulmonía (neumonía). O diagnóstico final é de bronquite aguda.

===== Resumen en japonés =====


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


45歳の女性で、気管支喘息の既往歴があり、3日間の咳と発熱のため受診されました。胸部レントゲン検査で肺炎の影は見られなかったため、肺炎は除外されました。診断は急性気管支炎です。

===== Resumen en ruso =====


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Вот краткая выписка из вашей медицинской карты:

Вы обратились к врачу с жалобами на кашель и температуру в течение трех дней, при этом у вас в анамнезе имеется бронхиальная астма. На основе осмотра и рентгена грудной клетки (который исключил воспаление легких) вам был поставлен диагноз: острый бронхит.

===== Resumen en árabe =====


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


إليك ملخص حالتك الصحية بناءً على التقارير الطبية:

حضرتِ إلى الاستشارة مع شكاوى من السعال والحرارة لمدة ثلاثة أيام، ولديك تاريخ سابِق من الربو الشعبي. أظهرت أشعة الصدر عدم وجود أي تكتلات، وتم استبعاد الإصابة بالالتهاب الرئوي. التشخيص السريري لحالتك هو التهاب القصبات الحاد.



---
## Bloque 3 · Temperatura

En la presentación vimos que el modelo no elige una palabra: calcula una **distribución de probabilidad** sobre el siguiente token y luego muestrea de ella. La **temperatura** controla cuánto se aplana esa distribución:

- **Temperatura baja (≈ 0):** casi siempre se elige el token más probable. Respuestas estables y repetibles.
- **Temperatura alta (≈ 1–1.5):** se eligen con más frecuencia tokens menos probables. Respuestas más variadas, y también más propensas a errores.

Vamos a lanzar **cinco veces el mismo prompt** con dos temperaturas y comparar las respuestas. Usamos `.batch()`, que envía varias peticiones a la vez (lo veremos en detalle en el bloque 5).

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

plantilla_titulo = ChatPromptTemplate.from_messages([
    ("human", "Escribe un título de menos de diez palabras para este informe clínico. "
              "Responde solo con el título.\n\n{informe}"),
])

N_REPETICIONES = 5
TEMPERATURAS = [0.0, 1.5]
entrada = {"informe": INFORMES[4]["texto"]}

resultados = {}
for t in TEMPERATURAS:
    cadena = plantilla_titulo | crear_modelo(temperatura=t)
    respuestas = cadena.batch([entrada] * N_REPETICIONES, config={"max_concurrency": 2})

    # Extraemos el contenido de forma segura si viene como lista
    textos_procesados = []
    for r in respuestas:
        contenido = r.content
        if isinstance(contenido, list) and len(contenido) > 0:
            if isinstance(contenido[0], dict) and 'text' in contenido[0]:
                contenido = contenido[0]['text']
        textos_procesados.append(str(contenido).strip())

    resultados[f"T = {t}"] = textos_procesados

tabla_temp = pd.DataFrame(resultados, index=[f"Ejecución {i+1}" for i in range(N_REPETICIONES)])
display(tabla_temp)
for col in tabla_temp:
    print(f"{col}: {tabla_temp[col].nunique()} respuestas distintas de {N_REPETICIONES}")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_

,T = 0.0,T = 1.5
Ejecución 1,Descompensación de insuficiencia cardiaca en paciente con fibrilación auricular,Descompensación de insuficiencia cardiaca en paciente con fibrilación auricular
Ejecución 2,Descompensación de insuficiencia cardiaca en paciente anticoagulada,Descompensación de insuficiencia cardiaca en mujer con fibrilación auricular
Ejecución 3,Insuficiencia cardiaca descompensada en mujer con fibrilación auricular,Descompensación de insuficiencia cardiaca en paciente anticoagulada
Ejecución 4,Insuficiencia cardiaca descompensada en mujer con fibrilación auricular,Descompensación de insuficiencia cardiaca en paciente con fibrilación auricular
Ejecución 5,Insuficiencia cardiaca descompensada en mujer con fibrilación auricular,Descompensación de insuficiencia cardiaca en paciente con fibrilación auricular


T = 0.0: 3 respuestas distintas de 5
T = 1.5: 3 respuestas distintas de 5


✏️ **Prueba tú.**

1. Cambia el prompt para que la tarea sea **extraer** algo concreto (por ejemplo, "¿qué fármaco toma la paciente?"). ¿Se nota igual la diferencia entre temperaturas?
2. Sube la temperatura a `2.0` (máximo permitido). ¿Siguen siendo correctas las respuestas?
3. ¿Obtienes siempre la misma respuesta con `T = 0`? La temperatura 0 no garantiza un resultado idéntico en todas las ejecuciones; anota lo que observes para las preguntas finales.

**Para pensar:** ¿qué temperatura usarías para extraer diagnósticos de un informe? ¿Y para generar un texto educativo para pacientes?

In [ ]:
def comparar_temperaturas(plantilla, entrada, temperaturas, n=5, concurrencia=2):
    resultados = {}
    for t in temperaturas:
        cadena = plantilla | crear_modelo(temperatura=t)
        respuestas = cadena.batch([entrada] * n, config={"max_concurrency": concurrencia})

        # Extraemos el contenido de forma segura si viene como lista
        textos_procesados = []
        for r in respuestas:
            contenido = r.content
            if isinstance(contenido, list) and len(contenido) > 0:
                if isinstance(contenido[0], dict) and 'text' in contenido[0]:
                    contenido = contenido[0]['text']
            textos_procesados.append(str(contenido).strip())

        resultados[f"T = {t}"] = textos_procesados

    tabla = pd.DataFrame(resultados, index=[f"Ejecución {i+1}" for i in range(n)])
    display(tabla)
    for col in tabla:
        print(f"{col}: {tabla[col].nunique()} respuestas distintas de {n}")
    return tabla

In [ ]:
plantilla_farmaco = ChatPromptTemplate.from_messages([
    ("human", "Según el informe, ¿qué fármaco o tipo de tratamiento toma la paciente? "
              "Responde solo con la respuesta, en una frase corta.\n\n{informe}"),
])

tabla_farmaco = comparar_temperaturas(plantilla_farmaco, {"informe": INFORMES[4]["texto"]}, [0.0, 1.5])

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_

,T = 0.0,T = 1.5
Ejecución 1,Tratamiento anticoagulante,Tratamiento anticoagulante.
Ejecución 2,Tratamiento anticoagulante.,Anticoagulantes.
Ejecución 3,Tratamiento anticoagulante.,Tratamiento anticoagulante.
Ejecución 4,Anticoagulantes.,Anticoagulantes.
Ejecución 5,Anticoagulantes.,Tratamiento anticoagulante


T = 0.0: 3 respuestas distintas de 5
T = 1.5: 3 respuestas distintas de 5


In [ ]:
# Prompt del título con tres temperaturas
tabla_titulo_2 = comparar_temperaturas(plantilla_titulo, {"informe": INFORMES[4]["texto"]}, [0.0, 1.5, 2.0])

# Prompt de extracción con T = 2.0
tabla_farmaco_2 = comparar_temperaturas(plantilla_farmaco, {"informe": INFORMES[4]["texto"]}, [0.0, 2.0])

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_

,T = 0.0,T = 1.5,T = 2.0
Ejecución 1,Ingreso por descompensación de insuficiencia cardiaca en fibrilación auricular,Descompensación de insuficiencia cardiaca en mujer anticoagulada,Descompensación de insuficiencia cardiaca en mujer anticoagulada.
Ejecución 2,Ingreso hospitalario por descompensación de insuficiencia cardiaca crónica.,Ingreso hospitalario por descompensación de insuficiencia cardiaca crónica,Descompensación de insuficiencia cardiaca en mujer anticoagulada
Ejecución 3,Ingreso por descompensación de insuficiencia cardiaca en fibrilación auricular,Descompensación de insuficiencia cardiaca en mujer con fibrilación auricular,Descompensación de insuficiencia cardiaca en paciente anticoagulada
Ejecución 4,Ingreso por descompensación de insuficiencia cardiaca en fibrilación auricular,Ingreso hospitalario por descompensación de insuficiencia cardiaca crónica,Descompensación de insuficiencia cardiaca en mujer con fibrilación auricular
Ejecución 5,Insuficiencia cardiaca descompensada en mujer con fibrilación auricular,Insuficiencia cardiaca descompensada en mujer con fibrilación auricular,Descompensación de insuficiencia cardiaca en mujer anticoagulada


T = 0.0: 3 respuestas distintas de 5
T = 1.5: 4 respuestas distintas de 5
T = 2.0: 4 respuestas distintas de 5


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_

,T = 0.0,T = 2.0
Ejecución 1,Tratamiento anticoagulante.,Tratamiento anticoagulante.
Ejecución 2,Tratamiento anticoagulante,Tratamiento anticoagulante
Ejecución 3,Tratamiento anticoagulante.,Anticoagulantes.
Ejecución 4,Tratamiento anticoagulante,Tratamiento anticoagulante.
Ejecución 5,Anticoagulantes.,Tratamiento anticoagulante


T = 0.0: 3 respuestas distintas de 5
T = 2.0: 3 respuestas distintas de 5


In [ ]:
tabla_t0 = comparar_temperaturas(plantilla_titulo, {"informe": INFORMES[4]["texto"]}, [0.0], n=10)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_

,T = 0.0
Ejecución 1,Ingreso por descompensación de insuficiencia cardiaca en mujer anticoagulada
Ejecución 2,Descompensación de insuficiencia cardiaca en mujer con fibrilación auricular
Ejecución 3,Ingreso por descompensación de insuficiencia cardiaca en fibrilación auricular
Ejecución 4,Descompensación de insuficiencia cardiaca en mujer con fibrilación auricular
Ejecución 5,Insuficiencia cardiaca descompensada en mujer con fibrilación auricular
Ejecución 6,Descompensación de insuficiencia cardiaca en mujer con fibrilación auricular
Ejecución 7,Descompensación de insuficiencia cardiaca en paciente anticoagulada
Ejecución 8,Ingreso por descompensación de insuficiencia cardiaca en mujer anticoagulada
Ejecución 9,Descompensación de insuficiencia cardiaca en paciente anticoagulada.
Ejecución 10,Ingreso por descompensación de insuficiencia cardiaca en mujer anticoagulada


T = 0.0: 6 respuestas distintas de 10


---
## Bloque 4 · Salida estructurada

Hasta ahora el modelo devolvía texto libre, que es cómodo de leer pero difícil de procesar: no se puede guardar en una base de datos, ni contar, ni comparar con una anotación. Para integrar un LLM en un sistema de información clínica necesitamos que devuelva **datos con un formato fijo**.

Para ello definimos un **esquema** con [Pydantic](https://docs.pydantic.dev): una clase de Python que describe qué campos queremos y de qué tipo son. Las descripciones de los campos (`description=...`) también forman parte del prompt: el modelo las lee para saber qué poner en cada campo.

In [ ]:
from pydantic import BaseModel, Field

class Diagnostico(BaseModel):
    diagnostico: str = Field(description="Nombre de la enfermedad o diagnóstico, tal como aparece en el texto")
    negado: bool = Field(description="True si el texto indica que el diagnóstico se descarta o no está presente")
    antecedente_familiar: bool = Field(description="True si el diagnóstico corresponde a un familiar y no al paciente")

class ExtraccionDiagnosticos(BaseModel):
    diagnosticos: list[Diagnostico] = Field(description="Todos los diagnósticos mencionados en el informe")

Con `with_structured_output` le pedimos al modelo que su respuesta cumpla el esquema. LangChain se encarga de enviar el esquema al proveedor y de convertir la respuesta en un objeto Python.

In [ ]:
INSTRUCCIONES_EXTRACCION = (
    "Eres un experto en documentación clínica. Extrae del informe todas las enfermedades y diagnósticos "
    "mencionados, tanto del paciente como de sus familiares, incluidos los que se descartan. "
    "No incluyas síntomas aislados, signos de la exploración ni resultados de pruebas."
)

plantilla_extraccion = ChatPromptTemplate.from_messages([
    ("system", INSTRUCCIONES_EXTRACCION),
    ("human", "Informe:\n{informe}"),
])

llm_estructurado = llm.with_structured_output(ExtraccionDiagnosticos)

informe = INFORMES[0]
resultado = llm_estructurado.invoke(plantilla_extraccion.invoke({"informe": informe["texto"]}))

print("Tipo de objeto:", type(resultado).__name__, "\n")
print(informe["texto"], "\n")
display(pd.DataFrame([d.model_dump() for d in resultado.diagnosticos]))
# model_dump() es un método de Pydantic que convierte un objeto Pydantic en un diccionario de Python

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Tipo de objeto: ExtraccionDiagnosticos 

Varón de 67 años con hipertensión arterial y diabetes mellitus tipo 2 que acude por dolor torácico. ECG sin alteraciones y troponinas normales; se descarta infarto agudo de miocardio. Padre fallecido por cáncer de colon. 



,diagnostico,negado,antecedente_familiar
0,hipertensión arterial,False,False
1,diabetes mellitus tipo 2,False,False
2,infarto agudo de miocardio,True,False
3,cáncer de colon,False,True


El resultado ya no es texto: es un objeto con atributos que podemos recorrer, filtrar o guardar. Por ejemplo, la lista de problemas **activos del paciente** son los diagnósticos no negados y no familiares:

In [ ]:
activos = [d.diagnostico for d in resultado.diagnosticos if not d.negado and not d.antecedente_familiar]
print("Problemas activos del paciente:", activos)

Problemas activos del paciente: ['hipertensión arterial', 'diabetes mellitus tipo 2']


In [ ]:
# Y el objeto se puede convertir también directamente en JSON, listo para otro sistema
print(resultado.model_dump_json(indent=2))

{
  "diagnosticos": [
    {
      "diagnostico": "hipertensión arterial",
      "negado": false,
      "antecedente_familiar": false
    },
    {
      "diagnostico": "diabetes mellitus tipo 2",
      "negado": false,
      "antecedente_familiar": false
    },
    {
      "diagnostico": "infarto agudo de miocardio",
      "negado": true,
      "antecedente_familiar": false
    },
    {
      "diagnostico": "cáncer de colon",
      "negado": false,
      "antecedente_familiar": true
    }
  ]
}


Probemos con los informes más difíciles. El informe 7 contiene una **sospecha** (no confirmada) y un **antecedente familiar negado**; el informe 10 menciona la misma enfermedad en la paciente y en su madre.

In [ ]:
for informe in [INFORMES[6], INFORMES[9]]:
    print(f"===== Informe {informe['id']} =====\n{informe['texto']}\n")
    resultado = llm_estructurado.invoke(plantilla_extraccion.invoke({"informe": informe["texto"]}))
    display(pd.DataFrame([d.model_dump() for d in resultado.diagnosticos]))

===== Informe 7 =====
Varón de 59 años con EPOC y obesidad. Se solicita polisomnografía por sospecha de síndrome de apnea obstructiva del sueño. Sin antecedentes familiares de enfermedad cardiovascular.



/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


,diagnostico,negado,antecedente_familiar
0,EPOC,False,False
1,obesidad,False,False
2,síndrome de apnea obstructiva del sueño,True,False
3,enfermedad cardiovascular,True,True


===== Informe 10 =====
Mujer de 25 años que consulta por cefalea intensa. Exploración neurológica normal. Se descarta hemorragia subaracnoidea mediante TC craneal. Juicio clínico: migraña sin aura. Antecedentes familiares de migraña por línea materna.



/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


,diagnostico,negado,antecedente_familiar
0,hemorragia subaracnoidea,True,False
1,migraña sin aura,False,False
2,migraña,False,True


✏️ **Prueba tú.**

1. **Añade un campo al esquema** `Diagnostico`, por ejemplo `sospecha: bool` ("True si el diagnóstico no está confirmado") o `familiar: str` ("parentesco del familiar, o cadena vacía"). Vuelve a ejecutar las celdas del bloque y comprueba cómo lo rellena con el informe 7.
2. Cambia la **descripción** del campo `negado` por algo vago, como `"negado"`. ¿Empeora la extracción?
3. Elimina de `INSTRUCCIONES_EXTRACCION` la frase sobre síntomas. ¿Aparece "dolor torácico" o "cefalea" como diagnóstico?

> Si modificas el esquema, recuerda volver a ejecutar la celda de `llm_estructurado = ...` para que el modelo use la nueva versión.

In [ ]:
class Diagnostico(BaseModel):
    diagnostico: str = Field(description="Nombre de la enfermedad o diagnóstico, tal como aparece en el texto")
    negado: bool = Field(description="True si el texto indica que el diagnóstico se descarta o no está presente")
    antecedente_familiar: bool = Field(description="True si el diagnóstico corresponde a un familiar y no al paciente")
    sospecha: bool = Field(description="True si el diagnóstico no está confirmado (sospecha, posible, a descartar)")
    familiar: str = Field(description="Parentesco del familiar (madre, padre, hermano...), o cadena vacía si es del paciente")

class ExtraccionDiagnosticos(BaseModel):
    diagnosticos: list[Diagnostico] = Field(description="Todos los diagnósticos mencionados en el informe")

# Hay que volver a crear el modelo estructurado con el esquema nuevo
llm_estructurado = llm.with_structured_output(ExtraccionDiagnosticos)

for informe in [INFORMES[6], INFORMES[9]]:
    print(f"===== Informe {informe['id']} =====\n{informe['texto']}\n")
    resultado = llm_estructurado.invoke(plantilla_extraccion.invoke({"informe": informe["texto"]}))
    display(pd.DataFrame([d.model_dump() for d in resultado.diagnosticos]))

===== Informe 7 =====
Varón de 59 años con EPOC y obesidad. Se solicita polisomnografía por sospecha de síndrome de apnea obstructiva del sueño. Sin antecedentes familiares de enfermedad cardiovascular.



/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


,diagnostico,negado,antecedente_familiar,sospecha,familiar
0,EPOC,False,False,False,
1,obesidad,False,False,False,
2,síndrome de apnea obstructiva del sueño,False,False,True,
3,enfermedad cardiovascular,True,True,False,familiares


===== Informe 10 =====
Mujer de 25 años que consulta por cefalea intensa. Exploración neurológica normal. Se descarta hemorragia subaracnoidea mediante TC craneal. Juicio clínico: migraña sin aura. Antecedentes familiares de migraña por línea materna.



/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


,diagnostico,negado,antecedente_familiar,sospecha,familiar
0,hemorragia subaracnoidea,True,False,False,
1,migraña sin aura,False,False,False,
2,migraña,False,True,False,madre


In [ ]:
class Diagnostico(BaseModel):
    diagnostico: str = Field(description="Nombre de la enfermedad o diagnóstico, tal como aparece en el texto")
    negado: bool = Field(description="negado")   # descripción vaga
    antecedente_familiar: bool = Field(description="True si el diagnóstico corresponde a un familiar y no al paciente")
    sospecha: bool = Field(description="True si el diagnóstico no está confirmado (sospecha, posible, a descartar)")
    familiar: str = Field(description="Parentesco del familiar (madre, padre, hermano...), o cadena vacía si es del paciente")

class ExtraccionDiagnosticos(BaseModel):
    diagnosticos: list[Diagnostico] = Field(description="Todos los diagnósticos mencionados en el informe")

llm_estructurado = llm.with_structured_output(ExtraccionDiagnosticos)

for informe in INFORMES[:3]:
    print(f"===== Informe {informe['id']} =====")
    resultado = llm_estructurado.invoke(plantilla_extraccion.invoke({"informe": informe["texto"]}))
    display(pd.DataFrame([d.model_dump() for d in resultado.diagnosticos]))

===== Informe 1 =====


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


,diagnostico,negado,antecedente_familiar,sospecha,familiar
0,hipertensión arterial,False,False,False,
1,diabetes mellitus tipo 2,False,False,False,
2,infarto agudo de miocardio,True,False,True,
3,cáncer de colon,False,True,False,padre


===== Informe 2 =====


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


,diagnostico,negado,antecedente_familiar,sospecha,familiar
0,asma bronquial,False,False,False,
1,neumonía,True,False,True,
2,bronquitis aguda,False,False,False,


===== Informe 3 =====


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


,diagnostico,negado,antecedente_familiar,sospecha,familiar
0,fibroadenoma de mama izquierda,False,False,False,
1,cáncer de mama,False,True,False,Madre
2,cáncer de mama,False,True,False,tía materna


In [ ]:
INSTRUCCIONES_SIN_SINTOMAS = (
    "Eres un experto en documentación clínica. Extrae del informe todas las enfermedades y diagnósticos "
    "mencionados, tanto del paciente como de sus familiares, incluidos los que se descartan."
)

plantilla_sin_sintomas = ChatPromptTemplate.from_messages([
    ("system", INSTRUCCIONES_SIN_SINTOMAS),
    ("human", "Informe:\n{informe}"),
])

for informe in [INFORMES[0], INFORMES[9]]:   # dolor torácico y cefalea
    print(f"===== Informe {informe['id']} =====")
    resultado = llm_estructurado.invoke(plantilla_sin_sintomas.invoke({"informe": informe["texto"]}))
    display(pd.DataFrame([d.model_dump() for d in resultado.diagnosticos]))

===== Informe 1 =====


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


,diagnostico,negado,antecedente_familiar,sospecha,familiar
0,hipertensión arterial,False,False,False,
1,diabetes mellitus tipo 2,False,False,False,
2,dolor torácico,False,False,False,
3,infarto agudo de miocardio,True,False,True,
4,cáncer de colon,False,True,False,padre


===== Informe 10 =====


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


,diagnostico,negado,antecedente_familiar,sospecha,familiar
0,cefalea intensa,False,False,False,
1,hemorragia subaracnoidea,True,False,True,
2,migraña sin aura,False,False,False,
3,migraña,False,True,False,madre


---
## Bloque 5 · Cadenas, procesamiento por lotes y evaluación

### 5.1 Una cadena `prompt | llm | parser`

LangChain permite unir piezas con el operador `|`: la salida de cada una es la entrada de la siguiente. Una cadena típica tiene tres eslabones:

1. **prompt**: rellena la plantilla con los datos.
2. **llm**: llama al modelo.
3. **parser**: convierte el texto de la respuesta en un objeto Python.

En el bloque 4, `with_structured_output` hacía los pasos 2 y 3 a la vez usando una función nativa del proveedor. Aquí usamos un `PydanticOutputParser` explícito, que añade al prompt unas instrucciones de formato y luego valida la respuesta contra el mismo esquema. Así se ven los tres eslabones por separado, igual que los verás después como tres cajas en Langflow.

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object=ExtraccionDiagnosticos)

plantilla_cadena = ChatPromptTemplate.from_messages([
    ("system", INSTRUCCIONES_EXTRACCION + "\n\n{instrucciones_formato}"),
    ("human", "Informe:\n{informe}"),
]).partial(instrucciones_formato=parser.get_format_instructions())

cadena = plantilla_cadena | llm | parser

# Una sola llamada
print(cadena.invoke({"informe": INFORMES[1]["texto"]}))

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


diagnosticos=[Diagnostico(diagnostico='asma bronquial', negado=False, antecedente_familiar=False, sospecha=False, familiar=''), Diagnostico(diagnostico='neumonía', negado=True, antecedente_familiar=False, sospecha=True, familiar=''), Diagnostico(diagnostico='bronquitis aguda', negado=False, antecedente_familiar=False, sospecha=False, familiar='')]


In [ ]:
# Las instrucciones de formato que el parser añade al prompt
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"$defs": {"Diagnostico": {"properties": {"diagnostico": {"description": "Nombre de la enfermedad o diagnóstico, tal como aparece en el texto", "title": "Diagnostico", "type": "string"}, "negado": {"description": "negado", "title": "Negado", "type": "boolean"}, "antecedente_familiar": {"description": "True si el diagnóstico corresponde a un familiar y no al paciente", "title": "Antecedente Familiar", "type": "boolean"}, "sospecha": {"description": "True si el diagnóstico no está confirmado (sospecha, posible, a descartar)", "title": "Sospecha"

### 5.2 Procesamiento por lotes con `.batch()`

`.batch()` ejecuta la cadena sobre una lista de entradas, con varias peticiones en paralelo. Con `return_exceptions=True` un error en un informe (por ejemplo, una respuesta que no cumple el esquema) no detiene el resto.

> Si usas el plan gratuito de Google y aparecen errores de límite de peticiones (*429, rate limit*), baja `max_concurrency` a 1.

In [ ]:
import time

entradas = [{"informe": inf["texto"]} for inf in INFORMES]

inicio = time.time()
predicciones = cadena.batch(entradas, config={"max_concurrency": 4}, return_exceptions=True)
print(f"{len(predicciones)} informes procesados en {time.time() - inicio:.1f} s")

errores = [inf["id"] for inf, p in zip(INFORMES, predicciones) if isinstance(p, Exception)]
print("Informes con error:", errores or "ninguno")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_

10 informes procesados en 42.2 s
Informes con error: ninguno


✏️ **Prueba tú.**

1. **Muestra el resultado** de uno de los informes. Recuerda que sigue la estructura Pydantic definida anteriormente.

In [ ]:
# Resultado del informe 3 (índice 2) como objeto Pydantic
resultado = predicciones[2]
print(type(resultado).__name__)
print(INFORMES[2]["texto"], "\n")

# Como tabla
display(pd.DataFrame([d.model_dump() for d in resultado.diagnosticos]))

# Como JSON
print(resultado.model_dump_json(indent=2))

ExtraccionDiagnosticos
Mujer de 52 años con fibroadenoma de mama izquierda diagnosticado por ecografía hace dos años, estable en el control actual. Madre y tía materna con cáncer de mama. 



,diagnostico,negado,antecedente_familiar,sospecha,familiar
0,fibroadenoma de mama izquierda,False,False,False,
1,cáncer de mama,False,True,False,madre
2,cáncer de mama,False,True,False,tía materna


{
  "diagnosticos": [
    {
      "diagnostico": "fibroadenoma de mama izquierda",
      "negado": false,
      "antecedente_familiar": false,
      "sospecha": false,
      "familiar": ""
    },
    {
      "diagnostico": "cáncer de mama",
      "negado": false,
      "antecedente_familiar": true,
      "sospecha": false,
      "familiar": "madre"
    },
    {
      "diagnostico": "cáncer de mama",
      "negado": false,
      "antecedente_familiar": true,
      "sospecha": false,
      "familiar": "tía materna"
    }
  ]
}


### 5.3 ¿Cuántos diagnósticos acierta?

Comparamos cada predicción con la anotación de referencia. Para cada diagnóstico anotado buscamos en la predicción uno con el mismo nombre (admitimos pequeñas diferencias de redacción y los sinónimos definidos al principio) y comprobamos sus dos atributos.

Clasificamos cada diagnóstico en:

- ✅ **correcto**: encontrado y con `negado` y `antecedente_familiar` bien.
- ⚠️ **atributo erróneo**: encontrado, pero con algún atributo mal. Es el error más peligroso: el sistema diría, por ejemplo, que el paciente tiene una neumonía que se ha descartado.
- ❌ **omitido**: el modelo no lo ha extraído.
- ➕ **sobrante**: el modelo extrae algo que no está en la referencia (un síntoma, una prueba o un diagnóstico inventado).

No hace falta que entiendas el código de evaluación en detalle: lo importante es la tabla de resultados.

In [ ]:
import unicodedata
from difflib import SequenceMatcher

def normalizar(texto):
    texto = unicodedata.normalize("NFKD", texto.lower()).encode("ascii", "ignore").decode()
    return " ".join(texto.replace(",", " ").replace(".", " ").split())

def similitud(pred, gold):
    # Puntuación entre 0 y 1 del parecido entre un nombre predicho y uno de referencia (o sus sinónimos)
    p = normalizar(pred)
    mejor = 0.0
    for nombre in [gold] + SINONIMOS.get(gold, []):
        g = normalizar(nombre)
        if p == g:
            return 1.0
        if g in p or p in g:
            mejor = max(mejor, 0.9)
        mejor = max(mejor, SequenceMatcher(None, p, g).ratio())
    return mejor

def evaluar_informe(informe, prediccion, umbral=0.8):
    gold = informe["gold"]
    pred = [] if isinstance(prediccion, Exception) else [d.model_dump() for d in prediccion.diagnosticos]

    # Emparejamiento voraz: primero las parejas más parecidas; a igual nombre, las que coinciden en atributos
    parejas = []
    for i, g in enumerate(gold):
        for j, p in enumerate(pred):
            s = similitud(p["diagnostico"], g["diagnostico"])
            if s >= umbral:
                coinciden = (p["negado"] == g["negado"]) + (p["antecedente_familiar"] == g["antecedente_familiar"])
                parejas.append((s + 0.01 * coinciden, i, j))
    parejas.sort(reverse=True)
    gold_usado, pred_usado, filas = {}, set(), []
    for _, i, j in parejas:
        if i not in gold_usado and j not in pred_usado:
            gold_usado[i] = j
            pred_usado.add(j)

    for i, g in enumerate(gold):
        fila = {"informe": informe["id"], "referencia": g["diagnostico"],
                "neg_ref": g["negado"], "fam_ref": g["antecedente_familiar"]}
        if i in gold_usado:
            p = pred[gold_usado[i]]
            ok = p["negado"] == g["negado"] and p["antecedente_familiar"] == g["antecedente_familiar"]
            fila.update(prediccion=p["diagnostico"], neg_pred=p["negado"], fam_pred=p["antecedente_familiar"],
                        resultado="✅ correcto" if ok else "⚠️ atributo erróneo")
        else:
            fila.update(prediccion="", neg_pred=None, fam_pred=None, resultado="❌ omitido")
        filas.append(fila)
    for j, p in enumerate(pred):
        if j not in pred_usado:
            filas.append({"informe": informe["id"], "referencia": "", "neg_ref": None, "fam_ref": None,
                          "prediccion": p["diagnostico"], "neg_pred": p["negado"], "fam_pred": p["antecedente_familiar"],
                          "resultado": "➕ sobrante"})
    return filas

def evaluar(predicciones, mostrar=True):
    tabla = pd.DataFrame([f for inf, p in zip(INFORMES, predicciones) for f in evaluar_informe(inf, p)])
    conteo = tabla["resultado"].value_counts()
    n_gold = sum(len(inf["gold"]) for inf in INFORMES)
    correctos = conteo.get("✅ correcto", 0)
    if mostrar:
        print(f"Diagnósticos correctos: {correctos} de {n_gold} ({100 * correctos / n_gold:.0f} %)")
        for etiqueta in ["⚠️ atributo erróneo", "❌ omitido", "➕ sobrante"]:
            print(f"  {etiqueta}: {conteo.get(etiqueta, 0)}")
    return tabla

In [ ]:
tabla_zero_shot = evaluar(predicciones)

Diagnósticos correctos: 29 de 31 (94 %)
  ⚠️ atributo erróneo: 1
  ❌ omitido: 1
  ➕ sobrante: 2


Veamos solo los casos en los que el modelo se ha equivocado:

In [ ]:
display(tabla_zero_shot[tabla_zero_shot["resultado"] != "✅ correcto"])

,informe,referencia,neg_ref,fam_ref,prediccion,neg_pred,fam_pred,resultado
9,3,,None,None,cáncer de mama,False,True,➕ sobrante
15,5,,None,None,descompensación de su insuficiencia cardiaca,False,False,➕ sobrante
18,6,infección cutánea,True,False,,None,None,❌ omitido
23,8,diabetes gestacional,True,False,diabetes gestacional,False,False,⚠️ atributo erróneo


### 5.4 ¿Mejora el modelo si le damos ejemplos? (*few-shot*)

En la presentación vimos el **aprendizaje en contexto**: el modelo puede adaptar su comportamiento solo con ejemplos incluidos en el prompt, sin reentrenarlo. Hasta ahora hemos trabajado en modo *zero-shot* (solo instrucciones). Vamos a añadir dos ejemplos resueltos que muestren los casos difíciles y a comparar los resultados con la misma evaluación.

Los ejemplos **no** pertenecen a los 10 informes de evaluación: si lo fueran, estaríamos dándole las respuestas al modelo.

In [ ]:
EJEMPLOS = """
Ejemplos resueltos:

Informe: "Paciente de 60 años con artrosis de rodilla. Se descarta gota tras analizar el líquido sinovial. Hermano con artritis reumatoide."
Diagnósticos:
- artrosis de rodilla (negado: false, antecedente_familiar: false)
- gota (negado: true, antecedente_familiar: false)
- artritis reumatoide (negado: false, antecedente_familiar: true)

Informe: "Mujer de 40 años con hipotiroidismo. No hay antecedentes familiares de cáncer de tiroides. Refiere cansancio."
Diagnósticos:
- hipotiroidismo (negado: false, antecedente_familiar: false)
- cáncer de tiroides (negado: true, antecedente_familiar: true)
(El cansancio es un síntoma y no se incluye.)
"""

plantilla_few_shot = ChatPromptTemplate.from_messages([
    ("system", INSTRUCCIONES_EXTRACCION + "\n" + EJEMPLOS + "\n{instrucciones_formato}"),
    ("human", "Informe:\n{informe}"),
]).partial(instrucciones_formato=parser.get_format_instructions())

cadena_few_shot = plantilla_few_shot | llm | parser
predicciones_few_shot = cadena_few_shot.batch(entradas, config={"max_concurrency": 4}, return_exceptions=True)

tabla_few_shot = evaluar(predicciones_few_shot)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_

Diagnósticos correctos: 31 de 31 (100 %)
  ⚠️ atributo erróneo: 0
  ❌ omitido: 0
  ➕ sobrante: 2


In [ ]:
def resumen(tabla):
    return tabla["resultado"].value_counts()

comparacion = pd.DataFrame({"zero-shot": resumen(tabla_zero_shot), "few-shot": resumen(tabla_few_shot)}).fillna(0).astype(int)
display(comparacion)

display(tabla_few_shot[tabla_few_shot["resultado"] != "✅ correcto"])

,zero-shot,few-shot
resultado,,
⚠️ atributo erróneo,1,0
✅ correcto,29,31
❌ omitido,1,0
➕ sobrante,2,2


,informe,referencia,neg_ref,fam_ref,prediccion,neg_pred,fam_pred,resultado
9,3,,None,None,cáncer de mama,False,True,➕ sobrante
15,5,,None,None,insuficiencia cardiaca,False,False,➕ sobrante


✏️ **Prueba tú.**

1. Ejecuta de nuevo la evaluación *zero-shot* y *few-shot*. ¿Son los resultados idénticos aunque la temperatura sea 0? ¿Qué implica esto para comparar dos prompts con solo 10 informes?
2. Cambia los ejemplos de `EJEMPLOS`: quita el que tiene un antecedente familiar negado y comprueba qué pasa con el informe 7.
3. Si has añadido el campo `sospecha` en el bloque 4, ¿cómo lo trata la evaluación? (Pista: la referencia no lo tiene anotado.)
4. Si tienes acceso a los dos proveedores, cambia `PROVEEDOR`, vuelve a ejecutar el notebook y compara los modelos con la misma evaluación.

In [ ]:
for intento in range(3):
    print(f"===== Intento {intento+1} =====")
    pred_zs = cadena.batch(entradas, config={"max_concurrency": 4}, return_exceptions=True)
    print("Zero-shot:")
    evaluar(pred_zs)
    pred_fs = cadena_few_shot.batch(entradas, config={"max_concurrency": 4}, return_exceptions=True)
    print("Few-shot:")
    evaluar(pred_fs)
    print()

===== Intento 1 =====


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_

Zero-shot:
Diagnósticos correctos: 29 de 31 (94 %)
  ⚠️ atributo erróneo: 1
  ❌ omitido: 1
  ➕ sobrante: 2


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_

Few-shot:
Diagnósticos correctos: 30 de 31 (97 %)
  ⚠️ atributo erróneo: 0
  ❌ omitido: 1
  ➕ sobrante: 2

===== Intento 2 =====


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_

... (3 outputs left out)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_

Zero-shot:
Diagnósticos correctos: 29 de 31 (94 %)
  ⚠️ atributo erróneo: 1
  ❌ omitido: 1
  ➕ sobrante: 2


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_

Few-shot:
Diagnósticos correctos: 30 de 31 (97 %)
  ⚠️ atributo erróneo: 0
  ❌ omitido: 1
  ➕ sobrante: 2



In [ ]:
EJEMPLOS_REDUCIDOS = """
Ejemplos resueltos:

Informe: "Paciente de 60 años con artrosis de rodilla. Se descarta gota tras analizar el líquido sinovial. Hermano con artritis reumatoide."
Diagnósticos:
- artrosis de rodilla (negado: false, antecedente_familiar: false)
- gota (negado: true, antecedente_familiar: false)
- artritis reumatoide (negado: false, antecedente_familiar: true)
"""

plantilla_few_shot_reducida = ChatPromptTemplate.from_messages([
    ("system", INSTRUCCIONES_EXTRACCION + "\n" + EJEMPLOS_REDUCIDOS + "\n{instrucciones_formato}"),
    ("human", "Informe:\n{informe}"),
]).partial(instrucciones_formato=parser.get_format_instructions())

cadena_reducida = plantilla_few_shot_reducida | llm | parser
pred_reducida = cadena_reducida.batch(entradas, config={"max_concurrency": 4}, return_exceptions=True)
tabla_reducida = evaluar(pred_reducida)

# Informe 7 con los dos prompts
display(tabla_few_shot[tabla_few_shot["informe"] == 7])
display(tabla_reducida[tabla_reducida["informe"] == 7])

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_

Diagnósticos correctos: 30 de 31 (97 %)
  ⚠️ atributo erróneo: 0
  ❌ omitido: 1
  ➕ sobrante: 2


,informe,referencia,neg_ref,fam_ref,prediccion,neg_pred,fam_pred,resultado
19,7,EPOC,False,False,EPOC,False,False,✅ correcto
20,7,obesidad,False,False,obesidad,False,False,✅ correcto
21,7,síndrome de apnea obstructiva del sueño,False,False,síndrome de apnea obstructiva del sueño,False,False,✅ correcto
22,7,enfermedad cardiovascular,True,True,enfermedad cardiovascular,True,True,✅ correcto


,informe,referencia,neg_ref,fam_ref,prediccion,neg_pred,fam_pred,resultado
19,7,EPOC,False,False,EPOC,False,False,✅ correcto
20,7,obesidad,False,False,obesidad,False,False,✅ correcto
21,7,síndrome de apnea obstructiva del sueño,False,False,síndrome de apnea obstructiva del sueño,False,False,✅ correcto
22,7,enfermedad cardiovascular,True,True,enfermedad cardiovascular,True,True,✅ correcto


---
## Del notebook a Langflow

En la próxima sesión construiremos estos mismos pasos con componentes visuales. Esta es la correspondencia:

| En este notebook | En Langflow (sesión 2) |
|---|---|
| `HumanMessage` / entrada de texto | *Chat Input* |
| `ChatPromptTemplate` con `{variables}` | *Prompt Template* |
| `init_chat_model(..., temperature=...)` | *Language Model* (con su parámetro de temperatura) |
| `with_structured_output` + esquema Pydantic | Componente de salida estructurada |
| `prompt \| llm \| parser` | Componentes unidos por conexiones en el lienzo |
| `.batch()` + función `evaluar()` | — (fíjate en si lo echas de menos) |

---


## Entregable

Termina los ejercicios ✏️ que no te haya dado tiempo a hacer en clase y responde brevemente (3–6 líneas cada una) a las siguientes preguntas en la celda de abajo. Entrega el notebook **ejecutado**, con las salidas visibles.

1. **Ejemplos en el prompt.** ¿Qué cambió al pasar de *zero-shot* a *few-shot*? Indica en qué informes y en qué tipo de error se nota la diferencia.
2. **Tipos de error.** ¿Qué tipo de error comete más el modelo: omisiones, atributos erróneos (negación o antecedente familiar) o diagnósticos sobrantes? ¿Cuál te parece más grave en un contexto clínico y por qué?
3. **Temperatura.** ¿Qué temperatura elegirías para la extracción de diagnósticos y cuál para redactar información para pacientes? ¿Garantiza la temperatura 0 que el resultado sea siempre el mismo?
4. **Incertidumbre.** El informe 7 contiene una *sospecha* de apnea del sueño. ¿Cómo la ha representado el modelo con el esquema original? Propón una modificación del esquema y justifícala.
5. **Evaluación.** Nuestra evaluación usa 10 informes y compara nombres de forma aproximada. Señala dos limitaciones de este método y explica qué harías antes de usar un sistema así con informes reales de un hospital (piensa también en la protección de datos).

### Respuestas

1. Al pasar de zero-shot a few-shot, los diagnósticos correctos subieron de 29 a 31 de 31 (del 94 % al 100 %). La mejora se nota en dos informes: en el 6 el modelo dejó de omitir la infección cutánea negada, y en el 8 marcó correctamente como negada la diabetes gestacional. Ambos son errores relacionados con la negación, justo el caso que mostraban los ejemplos (“se descarta gota”). Los dos sobrantes (informes 3 y 5) no cambiaron, porque los ejemplos no trataban ese problema. Con solo 10 informes y un modelo no determinista, la mejora es una señal y no una prueba.
2. El error más frecuente en el zero-shot son los diagnósticos sobrantes (2 de 4), seguidos de una omisión y un atributo erróneo. Los sobrantes que aparecen son duplicados de diagnósticos ya extraídos (el cáncer de mama familiar y la insuficiencia cardiaca), no invenciones, así que son poco graves. El más grave es el atributo erróneo: en el informe 8 el sistema habría registrado una diabetes gestacional que el texto descarta, y eso podría llevar a controles o tratamientos innecesarios. Es peor que una omisión o un duplicado porque el resultado parece correcto a simple vista y nadie lo revisaría.
3. Usaría temperatura 0 para extraer diagnósticos (precisión y repetibilidad) y 0.3-0.7 para redactar información a pacientes (naturalidad sin perder fidelidad clínica).
La temperatura 0 no garantiza el mismo resultado: con el mismo prompt obtuve 6 respuestas distintas de 10, desde un punto final hasta cambios de contenido.
Se debe a los cálculos en paralelo con números en coma flotante, que pueden cambiar el token elegido cuando dos tienen probabilidades casi iguales.
4. Con el esquema original, la sospecha de apnea del sueño se representaba como un diagnóstico normal (negado=False), sin distinguirla de uno confirmado. Propongo añadir sospecha: bool (“True solo si el texto expresa sospecha o posibilidad, no si se descarta”): al hacerlo, el modelo marcó la apnea con sospecha=True, lo que permite excluirla de los problemas activos hasta que se confirme.
5. Limitaciones: la muestra es muy pequeña (10 informes sintéticos) y el emparejamiento por nombres es aproximado, lo que puede dar falsos errores. Antes de usarlo con datos reales validaría con muchos más informes anotados por varios clínicos y con revisión humana, y cumpliría el RGPD (anonimizar, modelo local o con garantías y evaluación de impacto).